# CynipsNet fine-tuning for the STM32N6570-DK (v2)

Fine-tunes the speech model extracted from the board's firmware on Indian-accented English and on
recordings made **through the board's own microphone**, then saves a float checkpoint. Quantization,
the ST Edge AI compile and the firmware build run on your PC with `cynipsnet_re/deploy/deploy.py`.

**Before you start**
1. *Runtime → Change runtime type → GPU* (T4 is enough).
2. In Google Drive, create the folder `MyDrive/cynipsnet/` and put in it:
   * `cynipsnet_v1_float.pt` — **required** (from `cynipsnet_re/out/`). Without it the notebook stops.
   * recordings from `datacollect/board_record.py`, zipped (`*.zip`, each with a `manifest.jsonl`).
     A folder or zip whose name contains `test` is used **only for testing** — record one separate session for that.
   * optional: a Common Voice English release from **Mozilla Data Collective** (`*.tar.gz`); since October 2025
     Common Voice is distributed there, not on Hugging Face. Clips with an Indian / South Asian accent are used.
3. Svarah (Indian-accented English, CC BY 4.0, 9.6 h): on https://huggingface.co/datasets/ai4bharat/Svarah click
   *Agree and access*, create a read token at https://huggingface.co/settings/tokens and add it to the Colab
   secrets (key icon on the left) as `HF_TOKEN`.

Weights derived from CynipsNet are CC BY-NC-SA 4.0 (non-commercial).

In [ ]:
!nvidia-smi -L
!pip -q install datasets soundfile
import torch
assert torch.cuda.is_available(), "Switch the runtime to GPU first"

## 1. Training library (identical to `cynipsnet_re/colab/cynipsnet_finetune.py`, tested on the PC)

In [ ]:
%%writefile cynipsnet_finetune.py
"""Fine-tune CynipsNet (extracted from the STM32N6 firmware) on accented and board-microphone speech.

Everything the Colab notebook runs lives here, so the same code can be smoke-tested on a PC.
The front-end reproduces the firmware (src/asr_engine/mel, norm): 64 Slaney log-mels, 20 ms Hann
window, 10 ms hop, n_fft 512, pre-emphasis 0.97, per-band normalization, int8 input grid.
Output: a float checkpoint for cynipsnet_re/deploy (quantization and firmware build run locally,
where ST Edge AI is installed).
"""
import io
import json
import math
import os
import random
import re
import tarfile
import time
import zipfile
from collections import defaultdict

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

SR = 16000
TOKENS = "_ ABCDEFGHIJKLMNOPQRSTUVWXYZ'"
TOKEN_ID = {c: i for i, c in enumerate(TOKENS)}
INPUT_SCALE = 0.0628561676
MAX_SECONDS = 20.0

# ----------------------------------------------------------------------------- model (cynipsnet_arch.py)
BLOCKS = [(256, 33, 5), (256, 39, 5), (256, 39, 5), (512, 51, 5), (512, 63, 5), (512, 75, 5)]


class TFSameConv1d(nn.Conv1d):
    """Strided conv with TensorFlow SAME padding, as the deployed NPU graph computes C1: for an even
    number of frames 15 frames on the left and 16 on the right (PyTorch's symmetric 16/16 would shift
    the output by one frame). Parameter names are those of nn.Conv1d."""

    def forward(self, x):
        k, s, t = self.kernel_size[0], self.stride[0], x.shape[-1]
        total = max((-(-t // s) - 1) * s + k - t, 0)
        return F.conv1d(F.pad(x, (total // 2, total - total // 2)), self.weight, self.bias, s, 0,
                        self.dilation, self.groups)


class SeparableConv(nn.Module):
    def __init__(self, cin, cout, kernel, stride=1, dilation=1):
        super().__init__()
        pad = dilation * (kernel - 1) // 2
        if stride > 1:
            self.dw = TFSameConv1d(cin, cin, kernel, stride=stride, groups=cin)
        else:
            self.dw = nn.Conv1d(cin, cin, kernel, padding=pad, dilation=dilation, groups=cin)
        self.pw = nn.Conv1d(cin, cout, 1)

    def forward(self, x):
        return self.pw(self.dw(x))


class Block(nn.Module):
    def __init__(self, cin, cout, kernel, repeats):
        super().__init__()
        self.subs = nn.ModuleList(SeparableConv(cin if i == 0 else cout, cout, kernel) for i in range(repeats))
        self.residual = nn.Conv1d(cin, cout, 1)

    def forward(self, x):
        y = x
        for i, sub in enumerate(self.subs):
            y = sub(y)
            if i < len(self.subs) - 1:
                y = torch.relu(y)
        return torch.relu(y + self.residual(x))


class CynipsNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = SeparableConv(64, 256, 33, stride=2)
        blocks, cin = [], 256
        for cout, kernel, repeats in BLOCKS:
            blocks.append(Block(cin, cout, kernel, repeats))
            cin = cout
        self.blocks = nn.Sequential(*blocks)
        self.c2 = SeparableConv(512, 512, 87, dilation=2)
        self.c3 = nn.Conv1d(512, 1024, 1)
        self.c4 = nn.Conv1d(1024, 29, 1)

    def forward(self, mel):
        x = torch.relu(self.c1(mel))
        x = self.blocks(x)
        x = torch.relu(self.c2(x))
        x = torch.relu(self.c3(x))
        return self.c4(x)


def load_baseline(path, device):
    """Load the extracted checkpoint; refuse to continue from random weights."""
    if not os.path.exists(path):
        raise FileNotFoundError("%s not found: upload cynipsnet_v1_float.pt first. Training from random "
                                "weights produces a model that outputs nothing." % path)
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    model = CynipsNet()
    model.load_state_dict(ckpt["state_dict"] if "state_dict" in ckpt else ckpt, strict=True)
    # Fingerprint of the extracted CynipsNet v1.0: depthwise biases are all zero after extraction.
    return model.to(device)


# ----------------------------------------------------------------------------- front-end (firmware mel.c / norm.c)
def _mel_bank():
    def hz_to_mel(hz):
        return np.where(hz < 1000.0, hz / (200.0 / 3.0), 15.0 + np.log(np.maximum(hz, 1e-12) / 1000.0) / (np.log(6.4) / 27.0))

    def mel_to_hz(m):
        return np.where(m < 15.0, m * (200.0 / 3.0), 1000.0 * np.exp((np.log(6.4) / 27.0) * (m - 15.0)))

    edges = mel_to_hz(np.linspace(hz_to_mel(0.0), hz_to_mel(8000.0), 66))
    bins = np.arange(257) * (8000.0 / 256)
    bank = np.zeros((64, 257))
    for b in range(64):
        lo, c, hi = edges[b], edges[b + 1], edges[b + 2]
        w = np.minimum((bins - lo) / (c - lo), (hi - bins) / (hi - c))
        bank[b] = np.where(w > 0, w * 2.0 / (hi - lo), 0.0)
    return torch.tensor(bank, dtype=torch.float32)


class FrontEnd(nn.Module):
    """Batched waveform -> int8-grid normalized log-mel, plus valid frame counts."""

    def __init__(self):
        super().__init__()
        self.register_buffer("bank", _mel_bank())
        n = torch.arange(320, dtype=torch.float64)
        self.register_buffer("window", (0.5 - 0.5 * torch.cos(2 * math.pi * n / 319)).float())

    def forward(self, wave, lengths):
        """wave: [B, N] float in [-1, 1) (PCM16 / 32768); lengths: samples. -> ([B, 64, T], frames)."""
        e = torch.cat([wave[:, :1], wave[:, 1:] - 0.97 * wave[:, :-1]], dim=1)
        frames = e.unfold(1, 320, 160) * self.window
        power = torch.fft.rfft(frames, n=512, dim=-1).abs().pow(2)
        logmel = torch.log(power @ self.bank.T + 2.0 ** -24).transpose(1, 2)  # [B, 64, T]
        nframes = torch.clamp((lengths - 320) // 160 + 1, min=1)
        t = torch.arange(logmel.shape[2], device=wave.device)[None, None, :]
        mask = (t < nframes[:, None, None]).float()
        count = nframes[:, None, None].float()
        mu = (logmel * mask).sum(2, keepdim=True) / count
        var = ((logmel - mu) ** 2 * mask).sum(2, keepdim=True) / count
        x = (logmel - mu) / torch.clamp(var.sqrt(), min=1e-6)
        # The NPU gets int8 codes of x / INPUT_SCALE: train on that grid (straight-through gradient).
        xq = torch.clamp(torch.round(x / INPUT_SCALE), -128, 127) * INPUT_SCALE
        x = x + (xq - x).detach()
        return x * mask, nframes


# ----------------------------------------------------------------------------- text
_ONES = ("zero one two three four five six seven eight nine ten eleven twelve thirteen fourteen "
         "fifteen sixteen seventeen eighteen nineteen").split()
_TENS = "_ _ twenty thirty forty fifty sixty seventy eighty ninety".split()


def number_words(n):
    """Integer to English words, the way the board's alphabet spells numbers (no digits)."""
    if n < 20:
        return _ONES[n]
    if n < 100:
        return _TENS[n // 10] + ("" if n % 10 == 0 else " " + _ONES[n % 10])
    if n < 1000:
        rest = n % 100
        return _ONES[n // 100] + " hundred" + ("" if rest == 0 else " and " + number_words(rest))
    for value, name in ((10 ** 9, "billion"), (10 ** 6, "million"), (1000, "thousand")):
        if n >= value:
            rest = n % value
            return number_words(n // value) + " " + name + ("" if rest == 0 else
                                                           (" and " if rest < 100 else " ") + number_words(rest))
    return str(n)


def ordinal_words(n):
    words = number_words(n).split()
    last = words[-1]
    special = dict(one="first", two="second", three="third", five="fifth", eight="eighth", nine="ninth",
                   twelve="twelfth")
    words[-1] = special.get(last, last[:-1] + "ieth" if last.endswith("y") else last + "th")
    return " ".join(words)


def normalize_text(text):
    """Model alphabet: A-Z, apostrophe, space; numbers spelled out (number_words)."""
    text = text.replace("’", "'").replace("‘", "'")
    import unicodedata
    # Fold accents (café -> cafe); any other non-ASCII mark (dashes, quotes) separates words.
    text = "".join(c if ord(c) < 128 else " " for c in unicodedata.normalize("NFKD", text)
                   if not unicodedata.combining(c))
    if re.search(r"\d", text):
        text = re.sub(r"(\d+)(st|nd|rd|th)\b", lambda m: " " + ordinal_words(int(m.group(1))) + " ", text,
                      flags=re.IGNORECASE)
        text = re.sub(r"\d+", lambda m: " " + number_words(int(m.group())) + " ", text)
    text = text.upper().replace("-", " ")
    text = re.sub(r"[^A-Z' ]+", " ", text)
    # An apostrophe is a letter only inside a word (IT'S); quote marks around words are not.
    text = re.sub(r"(?<![A-Z])'|'(?![A-Z])", " ", text)
    text = " ".join(text.split())
    return text or None


def encode(text):
    return [TOKEN_ID[c] for c in text]


# ----------------------------------------------------------------------------- data sources
class Item:
    __slots__ = ("pcm", "text", "speaker", "source")

    def __init__(self, pcm, text, speaker, source):
        self.pcm, self.text, self.speaker, self.source = pcm, text, speaker, source


def to_pcm16(audio, sr):
    """float or int array at sr -> int16 at 16 kHz."""
    audio = np.asarray(audio)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if audio.dtype != np.int16:
        audio = np.clip(audio.astype(np.float32), -1.0, 1.0)
    else:
        audio = audio.astype(np.float32) / 32768.0
    if sr != SR:
        from math import gcd
        from scipy.signal import resample_poly
        g = gcd(int(sr), SR)
        audio = resample_poly(audio, SR // g, int(sr) // g).astype(np.float32)
    return np.clip(np.round(audio * 32767.0), -32768, 32767).astype(np.int16)


def _keep(pcm, text):
    return text is not None and 0.5 <= len(pcm) / SR <= MAX_SECONDS


def load_librispeech(root, limit=None, seed=0, source="librispeech"):
    """A LibriSpeech-style directory (speaker/chapter/*.flac + *.trans.txt)."""
    import soundfile as sf
    rows = []
    for dirpath, _, files in os.walk(root):
        for f in files:
            if f.endswith(".trans.txt"):
                for line in open(os.path.join(dirpath, f), encoding="utf-8"):
                    uid, text = line.strip().split(" ", 1)
                    rows.append((os.path.join(dirpath, uid + ".flac"), text, uid.split("-")[0]))
    rows.sort()
    random.Random(seed).shuffle(rows)
    items = []
    for path, text, spk in rows[:limit] if limit else rows:
        pcm, sr = sf.read(path, dtype="int16")
        text = normalize_text(text)
        if _keep(pcm, text):
            items.append(Item(to_pcm16(pcm, sr), text, "ls_" + spk, source))
    return items


def load_svarah(token, limit=None):
    """ai4bharat/Svarah (CC BY 4.0, gated: accept its terms on Hugging Face and pass a read token).

    There is no speaker column; speaker identity is approximated by the metadata fingerprint."""
    import soundfile as sf
    from datasets import Audio, load_dataset
    ds = load_dataset("ai4bharat/Svarah", split="test", token=token)
    ds = ds.cast_column("audio_filepath", Audio(decode=False))
    meta = ["gender", "age-group", "primary_language", "native_place_state", "native_place_district",
            "highest_qualification", "job_category", "occupation_domain"]
    items = []
    for i, row in enumerate(ds):
        if limit and i >= limit:
            break
        audio = row["audio_filepath"]
        data = audio.get("bytes")
        pcm, sr = sf.read(io.BytesIO(data) if data else audio["path"], dtype="float32")
        text = normalize_text(row["text"])
        pcm = to_pcm16(pcm, sr)
        if _keep(pcm, text):
            spk = "svarah_" + "|".join(str(row.get(k, "")) for k in meta)
            items.append(Item(pcm, text, spk, "svarah"))
    return items


def load_common_voice(cv_dir, accent_regex=r"india|south asia", splits=("train", "dev", "test"), limit=None):
    """A Common Voice English release from Mozilla Data Collective, extracted: <cv_dir>/{clips,*.tsv}.

    Keeps clips whose self-reported accent matches accent_regex. Returns {split: [Item]} using
    Common Voice's own speaker-disjoint splits."""
    import csv
    out = {}
    for split in splits:
        tsv = os.path.join(cv_dir, split + ".tsv")
        if not os.path.exists(tsv):
            continue
        items = []
        with open(tsv, encoding="utf-8") as f:
            for row in csv.DictReader(f, delimiter="\t", quoting=csv.QUOTE_NONE):
                accent = (row.get("accents") or row.get("accent") or "").lower()
                if not re.search(accent_regex, accent):
                    continue
                text = normalize_text(row["sentence"])
                if text is None:
                    continue
                import torchaudio
                wav, sr = torchaudio.load(os.path.join(cv_dir, "clips", row["path"]))
                pcm = to_pcm16(wav.mean(0).numpy(), sr)
                if _keep(pcm, text):
                    items.append(Item(pcm, text, "cv_" + row["client_id"], "commonvoice"))
                if limit and len(items) >= limit:
                    break
        out[split] = items
    return out


def load_manifest(path, source=None):
    """manifest.jsonl from datacollect/board_record.py (audio path relative to the manifest)."""
    import soundfile as sf
    base = os.path.dirname(os.path.abspath(path))
    items = []
    for line in open(path, encoding="utf-8"):
        row = json.loads(line)
        pcm, sr = sf.read(os.path.join(base, row["audio"]), dtype="int16")
        text = normalize_text(row["text"])
        pcm = to_pcm16(pcm, sr)
        if _keep(pcm, text):
            items.append(Item(pcm, text, "rec_" + row.get("speaker", "unknown"), source or row.get("source", "rec")))
    return items


def split_by_speaker(items, dev=0.1, test=0.2, seed=0):
    """Speaker-disjoint train/dev/test; with a single speaker, split that speaker's takes instead."""
    speakers = sorted({it.speaker for it in items})
    if len(speakers) < 3:
        rng = random.Random(seed)
        idx = list(range(len(items)))
        rng.shuffle(idx)
        n_test, n_dev = int(len(idx) * test), int(len(idx) * dev)
        pick = lambda ids: [items[i] for i in ids]
        return pick(idx[n_test + n_dev:]), pick(idx[n_test:n_test + n_dev]), pick(idx[:n_test])
    rng = random.Random(seed)
    rng.shuffle(speakers)
    n_test = max(1, int(round(len(speakers) * test)))
    n_dev = max(1, int(round(len(speakers) * dev)))
    test_s, dev_s = set(speakers[:n_test]), set(speakers[n_test:n_test + n_dev])
    tr = [it for it in items if it.speaker not in test_s | dev_s]
    return tr, [it for it in items if it.speaker in dev_s], [it for it in items if it.speaker in test_s]


def describe(name, items):
    hours = sum(len(it.pcm) for it in items) / SR / 3600
    return "%-24s %6d utterances %6.2f h %4d speakers" % (name, len(items), hours, len({i.speaker for i in items}))


# ----------------------------------------------------------------------------- augmentation (batched, on device)
def _synthetic_rir(rt60, device):
    n = int(SR * min(rt60 * 1.2, 1.0))
    t = torch.arange(n, device=device) / SR
    rir = torch.randn(n, device=device) * torch.exp(-6.9 * t / rt60)
    rir[0] = 1.0
    return rir / rir.norm()


def augment(wave, lengths, p_noise=0.7, p_reverb=0.4, p_eq=0.5):
    """Gain, colored noise at 5-35 dB SNR, synthetic reverberation, mild band limiting: the far-field,
    small-MEMS-mic conditions the board sees."""
    b, n = wave.shape
    dev = wave.device
    out = []
    for i in range(b):
        x = wave[i, :lengths[i]]
        if random.random() < p_reverb:
            rir = _synthetic_rir(random.uniform(0.15, 0.8), dev)
            m = x.numel() + rir.numel() - 1
            nfft = 1 << (m - 1).bit_length()
            y = torch.fft.irfft(torch.fft.rfft(x, nfft) * torch.fft.rfft(rir, nfft), nfft)[:x.numel()]
            x = y * (x.norm() / (y.norm() + 1e-8))
        if random.random() < p_eq:
            # Small-MEMS-mic band: smooth high-pass and low-pass in the frequency domain.
            spec = torch.fft.rfft(x)
            f = torch.linspace(0, SR / 2, spec.numel(), device=dev)
            hp, lp = random.uniform(60, 250), random.uniform(3500, 7500)
            gain = torch.sigmoid((f - hp) / (hp / 4)) * torch.sigmoid((lp - f) / 400.0)
            x = torch.fft.irfft(spec * gain, x.numel())
        if random.random() < p_noise:
            noise = torch.randn_like(x)
            alpha = random.choice([0.0, 0.5, 1.0])  # white, between, pink-ish
            if alpha > 0:
                spec = torch.fft.rfft(noise)
                f = torch.arange(spec.numel(), device=dev).clamp(min=1).float()
                noise = torch.fft.irfft(spec / f ** (alpha / 2), noise.numel())
            snr = random.uniform(5, 35)
            noise = noise * (x.norm() / (noise.norm() + 1e-8)) * 10 ** (-snr / 20)
            x = x + noise
        x = x * 10 ** (random.uniform(-12, 6) / 20)
        out.append(torch.clamp(x, -1.0, 1.0))
    res = torch.zeros_like(wave)
    for i, x in enumerate(out):
        res[i, :x.numel()] = x
    return res


def spec_augment(x, frames, n_freq=2, f_width=10, n_time=2, t_width=40):
    for i in range(x.shape[0]):
        for _ in range(n_freq):
            w = random.randint(0, f_width)
            f0 = random.randint(0, 64 - w)
            x[i, f0:f0 + w, :] = 0
        T = int(frames[i])
        for _ in range(n_time):
            w = random.randint(0, min(t_width, max(1, T // 10)))
            t0 = random.randint(0, max(0, T - w))
            x[i, :, t0:t0 + w] = 0
    return x


# ----------------------------------------------------------------------------- batching, loss, decoding
def batches(items, max_seconds=240.0, shuffle=True, seed=0):
    """Length-sorted buckets so padding stays small; a batch holds about max_seconds of audio."""
    order = sorted(range(len(items)), key=lambda i: len(items[i].pcm))
    groups, cur, longest = [], [], 0
    for i in order:
        longest = max(longest, len(items[i].pcm))
        if cur and longest * (len(cur) + 1) / SR > max_seconds:
            groups.append(cur)
            cur, longest = [], len(items[i].pcm)
        cur.append(i)
    if cur:
        groups.append(cur)
    if shuffle:
        random.Random(seed).shuffle(groups)
    return groups


def collate(items, idx, device):
    pcms = [items[i].pcm for i in idx]
    n = max(len(p) for p in pcms)
    wave = torch.zeros(len(idx), n)
    for k, p in enumerate(pcms):
        wave[k, :len(p)] = torch.from_numpy(p.astype(np.float32) / 32768.0)
    lengths = torch.tensor([len(p) for p in pcms])
    targets = [torch.tensor(encode(items[i].text)) for i in idx]
    return wave.to(device), lengths.to(device), targets


def out_lengths(frames):
    return (frames + 1) // 2  # C1 stride 2 with same padding


def greedy(logits):
    """logits [29, T] -> text (collapse repeats, drop blanks)."""
    ids = logits.argmax(0).tolist()
    out, prev = [], -1
    for i in ids:
        if i != prev and i != 0:
            out.append(TOKENS[i])
        prev = i
    return " ".join("".join(out).split())


def wer(refs, hyps):
    import numpy as _np

    def dist(r, h):
        d = list(range(len(h) + 1))
        for i in range(1, len(r) + 1):
            prev, d[0] = d[0], i
            for j in range(1, len(h) + 1):
                cur = min(d[j] + 1, d[j - 1] + 1, prev + (r[i - 1] != h[j - 1]))
                prev, d[j] = d[j], cur
        return d[len(h)]
    errs = sum(dist(r.split(), h.split()) for r, h in zip(refs, hyps))
    return 100.0 * errs / max(1, sum(len(r.split()) for r in refs))


@torch.no_grad()
def evaluate(model, frontend, items, device, max_seconds=400.0):
    model.eval()
    refs, hyps = [], []
    for idx in batches(items, max_seconds, shuffle=False):
        wave, lengths, _ = collate(items, idx, device)
        x, frames = frontend(wave, lengths)
        logits = model(x)
        olen = out_lengths(frames)
        for k, i in enumerate(idx):
            refs.append(items[i].text)
            hyps.append(greedy(logits[k, :, :olen[k]]))
    return wer(refs, hyps), list(zip(refs, hyps))


def train(model, frontend, train_sets, dev_sets, device, epochs=12, lr=1e-4, max_seconds=240.0,
          weights=None, guard_set=None, guard_margin=1.5, log=print, out_path="cynipsnet_finetuned.pt",
          max_steps=None):
    """Fine-tune on a weighted mixture of train_sets ({name: [Item]}); select on the dev sets.

    weights: {name: sampling multiplier} (e.g. board recordings x3). guard_set: a (name, items) dev
    set, normally LibriSpeech, whose WER may rise by at most guard_margin points over the start."""
    weights = weights or {}
    pool = []
    for name, items in train_sets.items():
        reps = weights.get(name, 1.0)
        full, frac = int(reps), reps - int(reps)
        pool += items * full + random.sample(items, int(len(items) * frac))
    log("training pool: %d utterances, %.1f h per epoch" % (len(pool), sum(len(i.pcm) for i in pool) / SR / 3600))
    steps_per_epoch = len(batches(pool, max_seconds))
    total = steps_per_epoch * epochs
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    warm = min(500, total // 10 + 1)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(s, total) / total)))
    ctc = nn.CTCLoss(blank=0, zero_infinity=True)
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    def dev_report():
        res = {name: evaluate(model, frontend, items, device)[0] for name, items in dev_sets.items()}
        if guard_set:
            res[guard_set[0]] = evaluate(model, frontend, guard_set[1], device)[0]
        return res

    start = dev_report()
    log("before fine-tuning: " + ", ".join("%s %.2f%%" % kv for kv in start.items()))
    best, best_score, history, step = None, None, [dict(epoch=0, **start)], 0
    for epoch in range(1, epochs + 1):
        model.train()
        t0, losses = time.time(), []
        for idx in batches(pool, max_seconds, seed=epoch):
            wave, lengths, targets = collate(pool, idx, device)
            with torch.no_grad():
                wave = augment(wave, lengths)
            x, frames = frontend(wave, lengths)
            x = spec_augment(x, frames)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(x)
            logp = logits.float().log_softmax(1).permute(2, 0, 1)  # [T, B, C]
            loss = ctc(logp, torch.cat(targets).to(device), out_lengths(frames),
                       torch.tensor([len(t) for t in targets], device=device))
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            losses.append(loss.item())
            step += 1
            if max_steps and step >= max_steps:
                break
        res = dev_report()
        history.append(dict(epoch=epoch, loss=float(np.mean(losses)), **res))
        target = [res[n] for n in dev_sets]
        score = float(np.mean(target)) if target else 0.0
        guard_ok = (not guard_set) or res[guard_set[0]] <= start[guard_set[0]] + guard_margin
        log("epoch %2d  loss %.3f  %s  (%.0f s)%s" % (epoch, np.mean(losses),
            ", ".join("%s %.2f%%" % kv for kv in res.items()), time.time() - t0,
            "" if guard_ok else "  [guard: LibriSpeech degraded, not selectable]"))
        if guard_ok and (best_score is None or score < best_score):
            best_score = score
            best = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            torch.save(dict(state_dict=best, input_scale=INPUT_SCALE, epoch=epoch, dev=res,
                            note="CynipsNet v1.0 fine-tuned; weights derived from CC BY-NC-SA 4.0 CynipsNet"),
                       out_path)
            log("  saved best -> %s" % out_path)
        if max_steps and step >= max_steps:
            break
    if best is not None:
        model.load_state_dict(best)
    return history


def unpack(archive, dest):
    """Extract a .zip or .tar(.gz) of recordings or a Common Voice release; return dest."""
    os.makedirs(dest, exist_ok=True)
    if archive.endswith(".zip"):
        zipfile.ZipFile(archive).extractall(dest)
    else:
        with tarfile.open(archive) as t:
            t.extractall(dest)
    return dest


def find_manifests(root):
    return [os.path.join(d, f) for d, _, fs in os.walk(root) for f in fs if f == "manifest.jsonl"]

## 2. Settings

In [ ]:
DRIVE = "/content/drive/MyDrive/cynipsnet"
USE_SVARAH = True            # needs HF_TOKEN in the Colab secrets
COMMON_VOICE_ARCHIVE = None  # e.g. DRIVE + "/cv-corpus-en.tar.gz" from Mozilla Data Collective
CV_LIMIT = 6000              # accented clips per split at most
LIBRI_TRAIN_UTTS = 3000      # LibriSpeech train-clean-100 utterances mixed in so it does not forget
EPOCHS = 8
LR = 1e-4
MAX_SECONDS_PER_BATCH = 240  # lower it if the GPU runs out of memory
WEIGHTS = {"board": 3.0, "pc": 1.5, "librispeech": 0.4}  # how often each source is seen per epoch
OUT = DRIVE + "/cynipsnet_finetuned.pt"

## 3. Load the extracted model and check it before anything else

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import importlib, os, glob
import cynipsnet_finetune as cf
importlib.reload(cf)
device = torch.device("cuda")
model = cf.load_baseline(DRIVE + "/cynipsnet_v1_float.pt", device)
fe = cf.FrontEnd().to(device)

!test -d LibriSpeech/dev-clean || (wget -q https://www.openslr.org/resources/12/dev-clean.tar.gz && tar xzf dev-clean.tar.gz)
libri_dev = cf.load_librispeech("LibriSpeech/dev-clean", limit=400, seed=0)
w, ex = cf.evaluate(model, fe, libri_dev, device)
print("baseline LibriSpeech dev-clean WER: %.2f%%  (expected about 5%%)" % w, ex[:2])
assert w < 10, "This is not the extracted CynipsNet checkpoint, or the front-end is broken: stop here."

## 4. Data

In [ ]:
train, dev, test = {}, {}, {"librispeech": libri_dev}

if USE_SVARAH:
    from google.colab import userdata
    sv = cf.load_svarah(userdata.get("HF_TOKEN"))
    train["svarah"], dev["svarah"], test["svarah"] = cf.split_by_speaker(sv, dev=0.1, test=0.2)

if COMMON_VOICE_ARCHIVE:
    cf.unpack(COMMON_VOICE_ARCHIVE, "/content/cv")
    cv_dir = next(d for d, _, fs in os.walk("/content/cv") if "train.tsv" in fs)
    cv = cf.load_common_voice(cv_dir, limit=CV_LIMIT)
    train["commonvoice"], dev["commonvoice"], test["commonvoice"] = cv.get("train", []), cv.get("dev", []), cv.get("test", [])

# Recordings from datacollect/board_record.py: zips or folders under DRIVE with a manifest.jsonl
for k, archive in enumerate(sorted(glob.glob(DRIVE + "/*.zip"))):
    cf.unpack(archive, "/content/rec/%d_%s" % (k, os.path.basename(archive)[:-4]))
for mf in cf.find_manifests("/content/rec") + cf.find_manifests(DRIVE):
    items = cf.load_manifest(mf)
    for src in sorted({i.source for i in items}):
        part = [i for i in items if i.source == src]
        if "test" in mf.lower():
            test[src] = test.get(src, []) + part
        else:
            tr, dv, _ = cf.split_by_speaker(part, dev=0.1, test=0.0)
            train[src], dev[src] = train.get(src, []) + tr, dev.get(src, []) + dv

!test -d LibriSpeech/train-clean-100 || (wget -q https://www.openslr.org/resources/12/train-clean-100.tar.gz && tar xzf train-clean-100.tar.gz)
train["librispeech"] = cf.load_librispeech("LibriSpeech/train-clean-100", limit=LIBRI_TRAIN_UTTS, seed=1)

for name, sets in (("train", train), ("dev", dev), ("test", test)):
    for k, v in sets.items():
        if v:
            print(cf.describe("%s/%s" % (name, k), v))
train = {k: v for k, v in train.items() if v}
dev = {k: v for k, v in dev.items() if v}
test = {k: v for k, v in test.items() if v}

## 5. Baseline accuracy on every test set (the accent gap)

In [ ]:
baseline = {k: cf.evaluate(model, fe, v, device)[0] for k, v in test.items()}
for k, v in baseline.items():
    print("baseline %-14s WER %.2f%%" % (k, v))

## 6. Fine-tune (the best epoch is saved to Drive after every epoch)

In [ ]:
history = cf.train(model, fe, train, {k: v for k, v in dev.items() if k != "librispeech"}, device,
                   epochs=EPOCHS, lr=LR, max_seconds=MAX_SECONDS_PER_BATCH, weights=WEIGHTS,
                   guard_set=("librispeech", libri_dev[:200]), out_path=OUT)

## 7. Before / after on held-out test sets

In [ ]:
import json
best = torch.load(OUT, weights_only=False)
model.load_state_dict(best["state_dict"])
after = {k: cf.evaluate(model, fe, v, device)[0] for k, v in test.items()}
report = {k: dict(baseline=round(baseline[k], 2), finetuned=round(after[k], 2)) for k in test}
for k, r in report.items():
    print("%-14s baseline %6.2f%%  ->  fine-tuned %6.2f%%" % (k, r["baseline"], r["finetuned"]))
json.dump(dict(report=report, history=history, epoch=best["epoch"]), open(DRIVE + "/finetune_report.json", "w"), indent=1)
print("\nCheckpoint:", OUT)

## 8. Put it on the board (on your PC)
Download `cynipsnet_finetuned.pt` from Drive into `cynipsnet_re/`, then:
```
python deploy/deploy.py cynipsnet_finetuned.pt --name cynipsnet_ft1 --calib ..\data\LibriSpeech\dev-clean recordings\manifest.jsonl --eval-manifest recordings_test\manifest.jsonl
```
It quantizes to int8, reports the int8 WER through the firmware's streaming path, compiles with ST Edge AI,
builds and signs the firmware, and prints the flash command (bootloader, app **and** weights together).